## Read direct XBPM measurement pickle files

### Read files, extract nominal and measured positions
#### Average positions and calculate their respective errors.
#### Plot nominal and measured positions.

In [ ]:
def extract_positions(data) -> list:
    return [
        (np.mean(data['X_pos']), np.std(data['X_pos'])),
        (np.mean(data['Y_pos']), np.std(data['Y_pos']))
        ]

In [ ]:
import pickle
import re
import numpy as np
from pathlib import Path
import os

# Working directory.

beamline = "MNC"
ipass = 2

homedir = os.environ['HOME']
wdir = (f"{homedir}/XBPM/Measurements/Analysis/2026/2026-09-28/"
        + f"{beamline}/{beamline}_pass0{ipass}/data/xbpm_meas_pass0{ipass}")

target_dir = Path(wdir)

# Read file names.
pfiles = [
    p.name
    for p in target_dir.iterdir()
    if (
        p.is_file() and
        p.name.startswith("xbpm_acq_") and
        p.name.endswith(".pickle")
        )
    ]


# Assemble MNC 1 & 2 data.
xbpm1 = {}
xbpm2 = {}

for pf in pfiles:

    # Read the angles from filename.
    agx = int(re.search(r"agx:([-\d]+)", pf).group(1))
    agy = int(re.search(r"agy:([-\d]+)", pf).group(1))
    print(f"Processing file with angles: ({agx:>7.2f},  {agy:>7.2f}) ")

    filename = f"{target_dir}/{pf}"
    with open(filename, "rb") as f:
        xbpm = pickle.load(f)
        xbpm1[(agx, agy)] = extract_positions(xbpm[f'{beamline}1'])
        xbpm2[(agx, agy)] = extract_positions(xbpm[f'{beamline}2'])

        # print(f"\n###\n### KEYS:\n###\n {xbpm.keys()}\n")

        # print("\n###\n### Data:\n###\n\n")
        # for k, d in xbpm['MNC1'].items():
        #     print(f"{k}: {d}")



### Put data into numpy array structures.

In [ ]:
unitfactor = 1  #  e-3    # nm to µm
xbpmdist_1 = 15.740  # m
xbpmdist_2 = 19.590  # m

angles = sorted(xbpm1.keys())

# MNC 1 
xn1 = np.array([k[0] for k in angles]) * xbpmdist_1
yn1 = np.array([k[1] for k in angles]) * xbpmdist_1

xp1 = np.array([xbpm1[k][0][0] for k in angles]) * unitfactor
yp1 = np.array([xbpm1[k][1][0] for k in angles]) * unitfactor

offset_x = xbpm1[0, 0][0][0] * unitfactor
offset_y = xbpm1[0, 0][1][0] * unitfactor
xp1 -= offset_x
yp1 -= offset_y

sxp1 = np.array([xbpm1[k][0][1] for k in angles]) * unitfactor
syp1 = np.array([xbpm1[k][1][1] for k in angles]) * unitfactor

# MNC 2
xn2 = np.array([k[0] for k in angles]) * xbpmdist_2
yn2 = np.array([k[1] for k in angles]) * xbpmdist_2

xp2 = np.array([xbpm2[k][0][0] for k in angles]) * unitfactor
yp2 = np.array([xbpm2[k][1][0] for k in angles]) * unitfactor

offset_x = xbpm2[0, 0][0][0] * unitfactor
offset_y = xbpm2[0, 0][1][0] * unitfactor
xp2 -= offset_x
yp2 -= offset_y

sxp2 = np.array([xbpm2[k][0][1] for k in angles]) * unitfactor
syp2 = np.array([xbpm2[k][1][1] for k in angles]) * unitfactor


In [ ]:
rsize = 5

rxn2 = xn2.reshape(rsize, rsize)[1:4, 1:4]
rxp2 = xp2.reshape(rsize, rsize)[1:4, 1:4]
ryn2 = yn2.reshape(rsize, rsize)[1:4, 1:4]
ryp2 = yp2.reshape(rsize, rsize)[1:4, 1:4]

rsize = 5
s2h = np.mean((rxn2 - rxp2)**2)
s2v = np.mean((ryn2 - ryp2)**2)
s2 = np.sqrt(s2h + s2v)

print(f"Erro horizontal: {np.sqrt(s2h)}")
print(f"Erro vertical:   {np.sqrt(s2v)}")
print(f"Erro total:      {s2}")

In [ ]:
%matplotlib qt5

### Plot nominal and XBPM-measured grids.

In [ ]:
import matplotlib.pyplot as plt

# plt.plot(xn1, yn1, 'r+')

fig1, (ax1, axdiff1) = plt.subplots(1, 2, figsize=(12, 5))
fig2, (ax2, axdiff2) = plt.subplots(1, 2, figsize=(12, 5))

# Reference positions.
ax1.plot(xn1, yn1, 'r+')
ax2.plot(xn2, yn2, 'r+')

rsize = 5

ax1.errorbar(xp1, yp1, xerr=sxp1, yerr=syp1,
             fmt='o', color='blue',
             ecolor='lightblue', elinewidth=2,
             capsize=4)
ax1.set_title(f'XBPM 1 @ {beamline}')

ax2.errorbar(xp2, yp2, xerr=sxp2, yerr=syp2,
             fmt='o', color='darkgreen',
             ecolor='grey', elinewidth=2,
             capsize=4)
ax2.set_title(f'XBPM 2 @ {beamline}')

diff1 = np.sqrt(
    (xp1.reshape(rsize, rsize) - xn1.reshape(rsize, rsize))**2 +
    (yp1.reshape(rsize, rsize) - yn1.reshape(rsize, rsize))**2
    )
extent = [xn1.min(), xn1.max(), yn1.min(), yn1.max()]
axdiff1.imshow(diff1, cmap='viridis', aspect='auto', extent=extent)
axdiff1.set_title('Difference')
axdiff1.set_xlabel('x [µm]')
axdiff1.set_ylabel('y [µm]')

diff2 = np.sqrt(
    (xp2.reshape(rsize, rsize) - xn2.reshape(rsize, rsize))**2 +
    (yp2.reshape(rsize, rsize) - yn2.reshape(rsize, rsize))**2
    )
axdiff2.imshow(diff2, cmap='viridis', aspect='auto', extent=extent)
axdiff2.set_title('Difference')
axdiff2.set_xlabel('x [µm]')
axdiff2.set_ylabel('y [µm]')

cbar1 = fig1.colorbar(
    axdiff1.imshow(diff1, cmap='viridis', aspect='auto', extent=extent),
    ax=axdiff1
    )
cbar1.set_label('Difference [µm]')

cbar2 = fig2.colorbar(
    axdiff2.imshow(diff2, cmap='viridis', aspect='auto', extent=extent),
    ax=axdiff2
    )
cbar2.set_label('Difference [µm]')

# fig.tight_layout()
ax1.set_xlabel('x [µm]')
ax1.set_ylabel('y [µm]')
ax2.set_xlabel('x [µm]')
ax2.set_ylabel('y [µm]')
ax1.grid(True)
ax2.grid(True)
axdiff1.grid(True)
axdiff2.grid(True)

title = f"XBPM Measured Data @ {beamline}"
fig1.suptitle(title)

outdir = (
    f"{homedir}/XBPM/Measurements/Analysis/2026/2026-09-28/{beamline}/"
    f"{beamline}_pass{ipass:02d}"
    )
outfile1 = f"{outdir}/xbpm_meas_{beamline}_pass{ipass}_xbpm1.png"
outfile2 = f"{outdir}/xbpm_meas_{beamline}_pass{ipass}_xbpm2.png"
fig1.tight_layout()
fig2.tight_layout()
# fig1.savefig(outfile1)
# fig2.savefig(outfile2)
plt.show()